In [1]:
import json, re
from bs4 import BeautifulSoup

def rsc_payload(html):
    """Join all self.__next_f.push([1,"..."]) chunks into one decoded string."""
    soup = BeautifulSoup(html, "html.parser")
    chunks = []
    for script in soup.find_all("script"):
        text = script.string or ""
        for m in re.finditer(r'self\.__next_f\.push\(\[1,\s*(".*?")\]\)', text, re.S):
            chunks.append(json.loads(m.group(1)))   # un-escapes \" and \u003c etc.
    return "".join(chunks)

def find_html(node):
    """Recursively collect every dangerouslySetInnerHTML.__html string."""
    if isinstance(node, dict):
        if "dangerouslySetInnerHTML" in node:
            yield node["dangerouslySetInnerHTML"]["__html"]
        for v in node.values():
            yield from find_html(v)
    elif isinstance(node, list):
        for v in node:
            yield from find_html(v)

def get_section(html, name="Provenance"):
    payload = rsc_payload(html)
    start = payload.find('{"name":' + json.dumps(name))
    if start == -1:
        return None
    obj, _ = json.JSONDecoder().raw_decode(payload, start)  # parses exactly one object
    raw = "".join(find_html(obj["body"]))
    return BeautifulSoup(raw.replace("<br/>", "\n"), "html.parser").get_text().strip()

In [13]:

with open ("scraped_pages/prestige_head.html", "r", encoding="utf-8") as file: 
    html = file.read()
get_section(html)

'Raoul Blondiau, Brussels, Belgium by 1935–February 1, 1937; [Galerie Charles Ratton, Paris, 1937]; Lilly Daché, Paris, France and  Pound Ridge, NY, 1937–74'